# Free multilingual voice assistant (local LLM via Ollama) with a female and a male character
**Pipeline:** microphone -> faster-whisper (ASR) -> **local LLM through Ollama (free, no API key)** -> TTS in the same language, with Layla (female) or Omar (male).

Works in **Google Colab** (use a GPU runtime) and in **local Jupyter**. Run the cells top to bottom.

In [59]:
!pip install -q -U faster-whisper edge-tts ollama soundfile "numpy<2.3" nest_asyncio
# Local Jupyter only (needs PortAudio): uncomment
# %pip install -q sounddevice

In [60]:
import os, re, shutil, sys, time, queue, asyncio, subprocess
from base64 import b64decode

import numpy as np
import soundfile as sf
import edge_tts
import nest_asyncio
import ollama
from faster_whisper import WhisperModel
from IPython.display import Audio, Javascript, display

nest_asyncio.apply()                      # lets asyncio.run() work inside Jupyter
IN_COLAB = "google.colab" in sys.modules
SAMPLE_RATE = 16000

ALLOWED_LANGS = ["ar", "en"]   # languages you speak; add more e.g. ["ar","en","fr"]. None = all languages
FORCE_LANG = None             # e.g. "ar" to always assume Arabic (most reliable)
WHISPER_SIZE = "small"        # tiny / base / small / medium / large-v3 (large-v3 needs a GPU)
LLM_MODEL = "qwen3:8b"        # good multilingual incl. Arabic (~5 GB). Weak PC: "gemma3:4b". Strong GPU: "qwen3:14b"

## 0. Start Ollama and download the model (one time)
- **Colab:** this cell installs and starts Ollama for you (pick a GPU runtime: Runtime -> Change runtime type -> T4 GPU).
- **Local PC:** install Ollama from ollama.com first, then run this cell (it only pulls the model).

In [61]:
def ollama_up():
    try:
        ollama.list(); return True
    except Exception:
        return False

if not shutil.which("ollama"):
    if IN_COLAB:
        subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    else:
        raise SystemExit("Install Ollama from https://ollama.com, then re-run this cell.")

if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up(): break
        time.sleep(1)

print("Pulling", LLM_MODEL, "(first time only, a few GB)...")
ollama.pull(LLM_MODEL)
print("Ollama ready.")

Pulling qwen3:8b (first time only, a few GB)...
Ollama ready.


## 1. The two characters

In [62]:
CHARACTERS = {
    "female": {"name": "Layla", "gender": "Female",
               "persona": "You are Layla, a warm, cheerful and concise voice assistant.",
               "fallback_voice": "en-US-AvaMultilingualNeural"},
    "male":   {"name": "Omar", "gender": "Male",
               "persona": "You are Omar, a calm, friendly and concise voice assistant.",
               "fallback_voice": "en-US-AndrewMultilingualNeural"},
}
SWITCH_PHRASES = {   # spoken commands that switch character; add your own languages
    "female": ["female voice", "woman voice", "switch to layla", "صوت أنثى", "صوت انثى", "صوت امرأة"],
    "male":   ["male voice", "man voice", "switch to omar", "صوت ذكر", "صوت رجل"],
}
PREFERRED_LOCALE = {"ar": "ar-EG", "en": "en-US", "es": "es-ES", "fr": "fr-FR",
                    "pt": "pt-BR", "zh": "zh-CN", "de": "de-DE"}
state = {"char": "female"}

## 2. Microphone input
Colab records a fixed number of seconds through the browser. Local Jupyter records until you stop talking.

In [63]:
RECORD_JS = '''
const sleep = t => new Promise(r => setTimeout(r, t));
const b2text = blob => new Promise(r => { const f = new FileReader(); f.onloadend = e => r(e.srcElement.result); f.readAsDataURL(blob); });
var record = t => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream); const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start(); await sleep(t);
  rec.onstop = async () => { resolve(await b2text(new Blob(chunks))); };
  rec.stop(); stream.getTracks().forEach(x => x.stop());
});
'''

def record_colab(seconds=6):
    from google.colab import output
    display(Javascript(RECORD_JS))
    print(f"Recording {seconds}s - speak now...")
    data = output.eval_js(f"record({int(seconds * 1000)})")
    open("rec.webm", "wb").write(b64decode(data.split(",")[1]))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "rec.webm",
                    "-ar", str(SAMPLE_RATE), "-ac", "1", "rec.wav"], check=True)
    audio, _ = sf.read("rec.wav", dtype="float32")
    return audio

def record_local(max_seconds=30, silence_seconds=1.0, threshold=0.012):
    import sounddevice as sd
    q = queue.Queue()
    block = int(SAMPLE_RATE * 0.1)
    frames, started, silent = [], False, 0.0
    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="float32", blocksize=block,
                        callback=lambda indata, n, t, s: q.put(indata.copy())):
        t0 = time.time()
        while time.time() - t0 < max_seconds:
            chunk = q.get()
            rms = float(np.sqrt(np.mean(chunk ** 2)))
            if rms > threshold:
                started, silent = True, 0.0
            elif started:
                silent += block / SAMPLE_RATE
            if started:
                frames.append(chunk)
            if started and silent >= silence_seconds:
                break
    return np.concatenate(frames)[:, 0] if frames else None

def record(seconds=6):
    return record_colab(seconds) if IN_COLAB else record_local()

## 3. Speech recognition (auto-detects the language)

In [64]:
try:
    import ctranslate2
    HAS_CUDA = ctranslate2.get_cuda_device_count() > 0
except Exception:
    HAS_CUDA = False

def load_asr(size):
    # Try the GPU first and run a 1-second warm-up: missing CUDA libraries (libcublas.so.12) only
    # fail at the first transcription, so we test it here and fall back to CPU if it breaks.
    if HAS_CUDA:
        try:
            m = WhisperModel(size, device="cuda", compute_type="float16")
            segs, _ = m.transcribe(np.zeros(SAMPLE_RATE, dtype="float32"))
            list(segs)
            print(f"faster-whisper '{size}' running on GPU")
            return m
        except Exception as e:
            print(f"GPU not usable ({str(e)[:90]}...) -> using CPU instead")
    print(f"faster-whisper '{size}' running on CPU (int8)")
    return WhisperModel(size, device="cpu", compute_type="int8")

asr = load_asr(WHISPER_SIZE)

def pick_language(audio):
    # Whisper's free language guess is unreliable on short clips (Arabic often becomes Hindi/Urdu/Persian).
    # So we only choose among the languages you actually speak (ALLOWED_LANGS).
    if FORCE_LANG:
        return FORCE_LANG
    if not ALLOWED_LANGS:
        return None                                   # None = let Whisper decide among all languages
    try:
        _, _, probs = asr.detect_language(audio, vad_filter=True)
        p = dict(probs)
        best = max(ALLOWED_LANGS, key=lambda l: p.get(l, 0.0))
        print("[lang]", {l: round(p.get(l, 0.0), 2) for l in ALLOWED_LANGS}, "->", best)
        return best
    except Exception as e:
        print("[lang] detection failed:", str(e)[:80])
        return ALLOWED_LANGS[0]

def transcribe(audio):
    lang = pick_language(audio)
    segments, info = asr.transcribe(audio, language=lang, beam_size=5, vad_filter=True)
    text = " ".join(s.text.strip() for s in segments).strip()
    return text, (lang or info.language)

GPU not usable (Library libcublas.so.12 is not found or cannot be loaded...) -> using CPU instead
faster-whisper 'small' running on CPU (int8)


## 4. Speech synthesis (male / female voice in any language)
Edge neural voices cover ~100+ languages and have a male and female voice for most of them (needs internet).

In [65]:
_voices = None

async def _pick_voice(lang, gender, fallback):
    global _voices
    if _voices is None:
        _voices = await edge_tts.list_voices()
    cands = [v for v in _voices if v["Gender"] == gender and v["Locale"].split("-")[0] == lang]
    pref = PREFERRED_LOCALE.get(lang)
    if pref:
        cands.sort(key=lambda v: v["Locale"] != pref)
    return cands[0]["ShortName"] if cands else fallback

async def _synth(text, lang, char, path):
    voice = await _pick_voice(lang, char["gender"], char["fallback_voice"])
    print(f"[tts] voice = {voice}")
    await edge_tts.Communicate(text, voice).save(path)

USE_SCRATCH_TTS = False   # set True by the optional 'own voice' cell below

def speak(text, lang, char):
    if USE_SCRATCH_TTS and lang == 'en':
        return speak_scratch(text, char)
    path = "reply.mp3"
    asyncio.run(_synth(text, lang, char, path))
    display(Audio(filename=path, autoplay=True))

## 5. The brain (LLM)

In [66]:
history = []

LANG_NAMES = {"ar": "Arabic", "en": "English", "fr": "French", "es": "Spanish", "de": "German",
              "tr": "Turkish", "it": "Italian", "ru": "Russian", "zh": "Chinese", "ja": "Japanese",
              "hi": "Hindi", "ur": "Urdu", "pt": "Portuguese", "ko": "Korean"}

def lang_hint(lang):
    if not lang:
        return ""
    return f"The user is speaking {LANG_NAMES.get(lang, lang)}; reply in {LANG_NAMES.get(lang, lang)}."

def think(user_text, char, lang=None):
    system = (f"{char['persona']} {lang_hint(lang)} You are speaking aloud. Always answer in the same language the user "
              "spoke. Use short, natural spoken sentences. No markdown, no lists, no emojis.")
    history.append({"role": "user", "content": user_text})
    msgs = [{"role": "system", "content": system}] + history[-20:]
    opts = {"num_predict": 300, "temperature": 0.7}
    try:
        r = ollama.chat(model=LLM_MODEL, messages=msgs, think=False, options=opts)   # Qwen3: no long 'thinking'
    except TypeError:
        r = ollama.chat(model=LLM_MODEL, messages=msgs, options=opts)
    text = re.sub(r"<think>.*?</think>", "", r["message"]["content"], flags=re.S).strip()
    history.append({"role": "assistant", "content": text})
    return text

## 5b. (Optional) Use YOUR from-scratch voice
Only after you trained `tts_from_scratch.ipynb`: put `tts_scratch.py` (from the first answer) and the trained `tts.pt` next to this notebook, then run this cell. It speaks **English only**; other languages keep using Edge voices. Skip it otherwise.

In [ ]:
import torch
import tts_scratch as ts

_ck = torch.load("tts.pt", map_location="cpu")
_speakers = _ck["speakers"]
_scratch = ts.Tacotron(len(ts.SYMBOLS), len(_speakers))
_scratch.load_state_dict(_ck["model"]); _scratch.eval()

def speak_scratch(text, char):
    spk = "female" if char["gender"] == "Female" else "male"
    ids = torch.tensor([ts.text_to_ids(text)])
    mel = _scratch.infer(ids, torch.tensor([_speakers.index(spk)]))
    wav = ts.mel_to_wav(mel)
    sf.write("reply_scratch.wav", wav / max(1e-6, np.abs(wav).max()) * 0.9, ts.SR)
    display(Audio(filename="reply_scratch.wav", autoplay=True))

USE_SCRATCH_TTS = True

## 6. One conversation turn

In [67]:
def detect_switch(text):
    low = text.lower()
    for key, phrases in SWITCH_PHRASES.items():
        if any(p in low for p in phrases):
            return key
    return None

def handle(text, lang):
    # text -> (maybe switch character) -> LLM -> speak
    new = detect_switch(text)
    if new and new != state["char"]:
        state["char"] = new
        char = CHARACTERS[new]
        speak(f"Hi, I'm {char['name']}.", "en", char)
        return
    char = CHARACTERS[state["char"]]
    answer = think(text, char, lang)
    print(f"{char['name']}: {answer}")
    speak(answer, lang, char)

def talk(seconds=6):
    # speak into the mic, get a spoken answer
    audio = record(seconds)
    if audio is None or len(audio) == 0:
        print("(nothing heard)"); return
    text, lang = transcribe(audio)
    print(f"you ({lang}): {text}")
    if text:
        handle(text, lang)

def chat(text, lang="en"):
    # same thing without a microphone - handy for testing
    print(f"you ({lang}): {text}")
    handle(text, lang)

## 7. Try it

In [68]:
# Text test (no mic needed): same question, both characters
state["char"] = "female"
chat("What's the capital of Egypt and one fun fact about it?", "en")

you (en): What's the capital of Egypt and one fun fact about it?
Layla: The capital of Egypt is Cairo. A fun fact is that Cairo is one of the oldest cities in the world, with a history spanning over 9,000 years.
[tts] voice = en-US-AvaNeural


In [69]:
state["char"] = "male"
chat("ما هي أكبر مدينة في مصر؟", "ar")

you (ar): ما هي أكبر مدينة في مصر؟
Omar: أكبر مدينة في مصر هي القاهرة.
[tts] voice = ar-EG-ShakirNeural


In [77]:
# Voice: run this cell, then talk (Colab: allow the microphone in the browser; it records 6 s)
talk(seconds=6)

<IPython.core.display.Javascript object>

Recording 6s - speak now...
[lang] {'ar': 0.04, 'en': 0.2} -> en
you (en): may he have a permanent life


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [76]:
# Hands-free-ish loop (re-run the cell above for each turn, or use this; stop with the Interrupt button)
while True:
    talk(seconds=6)

<IPython.core.display.Javascript object>

Recording 6s - speak now...
[lang] {'ar': 0.0, 'en': 0.88} -> en
you (en): Can you hear me now?
Omar: Yes, I can hear you now. How can I assist you?
[tts] voice = en-US-AndrewNeural


<IPython.core.display.Javascript object>

Recording 6s - speak now...
[lang] {'ar': 0.02, 'en': 0.08} -> en
you (en): May I open my tinnabarra?
Omar: Sure, you can open your tinnabarra. Let me know if you need anything else.
[tts] voice = en-US-AndrewNeural


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Recording 6s - speak now...
[lang] {'ar': 0.9, 'en': 0.02} -> ar
you (ar): ما هي أكبر مدينة بلاد
Omar: القاهرة هي أكبر مدينة في مصر.
[tts] voice = ar-EG-ShakirNeural


<IPython.core.display.Javascript object>

Recording 6s - speak now...
[lang] {'ar': 0.03, 'en': 0.07} -> en


KeyboardInterrupt: 

### Notes
- Everything here is free: Whisper, Ollama + the model, and Edge voices (Edge needs internet). Nothing needs an API key.
- Say "male voice" / "female voice" to switch character, or set `state["char"] = "male"`.
- Change `LLM_MODEL` at the top to try another Ollama model (`ollama.pull` it first). Bigger model = smarter but slower.
- Lower latency: stream the reply (`stream=True`) and speak sentence by sentence.